# Q6. Receiving plant vs supplier region

Scope (from `eda/scope_eda`): purchasing org **US00**, purchase orders only (category F, deleted items excluded), Document Date **Oct 2025 – Sep 2026**, USD. Measure = **Net PO value** (ordered value, not invoiced or paid spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data and charts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sap_data import load_scope, product_family, FAMILY_CATEGORY
from charts import SERIES, MUTED, INK_2, usd, pct, style, save

style()
pd.set_option("display.max_columns", 30)
PERIOD = "Oct 2025 – Sep 2026"

w = load_scope()
w["supplier"] = w["Name 1"] + " (" + w["Supplier"].fillna("n/a") + ")"
TOTAL = w["Net Order Value"].sum()
print(f"{len(w)} PO items | {w['Purchasing Document'].nunique()} POs | {TOTAL:,.0f} USD")

**Question.** For each US receiving plant, what share of its Net PO value comes from suppliers in each
country/region, and does any plant depend on a single supplier region?

Plant = where the items are ordered **for** (EKPO). Region = supplier master address (LFA1): the US
state for domestic suppliers, the country for overseas ones. A plant is called **dependent** when one
region supplies more than 50% of its value.

In [ ]:
PLANTS = {"MI00": "Miami (MI00)", "DL00": "Dallas (DL00)", "SD00": "San Diego (SD00)"}
w["plant"] = w["Plant"].map(PLANTS).fillna("(no plant)")
w["country"] = w["Country/Region Key"].fillna("Unknown")
w["region"] = np.where(w["country"] == "US", "US-" + w["Region"].fillna("unknown state"), w["country"])
w.groupby("plant").agg(items=("Item", "size"), net=("Net Order Value", "sum"),
                       suppliers=("Supplier", "nunique")).style.format({"net": "{:,.0f}"})

In [ ]:
by_country = pd.crosstab(w["plant"], w["country"], values=w["Net Order Value"], aggfunc="sum",
                         normalize="index").fillna(0)
top = ["US", "DE", "HU"]
by_country = by_country[top].assign(Other=1 - by_country[top].sum(axis=1))
by_country.style.format("{:.1%}")

In [ ]:
main = ["Miami (MI00)", "Dallas (DL00)"]
d = by_country.loc[main[::-1]]
fig, ax = plt.subplots(figsize=(10, 2.6))
left = np.zeros(len(d))
for col, color in zip(d.columns, SERIES[:3] + [MUTED]):
    ax.barh(d.index, d[col], left=left, color=color, height=0.55, label=col, edgecolor="#fcfcfb", lw=2)
    for y, (l, v) in enumerate(zip(left, d[col])):
        if v > 0.04:
            ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    left += d[col].to_numpy()
ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 1); ax.grid(visible=False)
ax.legend(ncol=4, loc="upper left", bbox_to_anchor=(0, -0.18), title="Supplier country")
ax.set_title(f"Share of each plant's Net PO value by supplier country, {PERIOD}")
save(fig, "q6_plant_by_country")

In [ ]:
reg = (w.groupby(["plant", "region"])["Net Order Value"].sum()
         .groupby(level=0, group_keys=False).apply(lambda s: (s / s.sum()).sort_values(ascending=False)))
summary = (reg.groupby(level=0).agg(top_region=lambda s: s.index[0][1], top_share="first",
                                    regions="size", HHI=lambda s: (s ** 2).sum())
              .assign(dependent=lambda d: d.top_share > 0.5))
display(summary.style.format({"top_share": "{:.0%}", "HHI": "{:.2f}"}))
reg.groupby(level=0).head(5).rename("share").to_frame().style.format("{:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharex=True)
for ax, p in zip(axes, main):
    s = reg.loc[p].head(6).iloc[::-1]
    colors = [MUTED if "unknown" in r else SERIES[0] if r.startswith("US-") else SERIES[1] for r in s.index]
    ax.barh(s.index, s, color=colors, height=0.6)
    for y, v in enumerate(s):
        ax.text(v, y, f" {v:.0%}", va="center", color=INK_2, fontsize=9)
    ax.axvline(0.5, color=MUTED, ls="--", lw=1)
    ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 1); ax.set_title(p)
    ax.grid(axis="y", visible=False)
fig.suptitle("Top supplier regions per plant (blue = US state, orange = overseas, grey = no state); "
             "dashed = 50% dependence threshold", x=0.01, ha="left", fontweight="bold", fontsize=10.5)
fig.tight_layout()
save(fig, "q6_plant_top_regions")

## Findings for Antonio

**Finding.** Both main plants buy about **95% of their value from US suppliers**: Miami (MI00, $2.91M) 95.4% and Dallas (DL00, $139k) 94.9%. Germany and Hungary supply the rest. By region, Miami is spread out (largest region 40%, then Texas 24% and Nebraska 21%). **Dallas looks dependent: 91% comes from US suppliers with no state recorded.** San Diego has a single $1k item.

**Decision it supports.** At country level, both plants depend on domestic supply. Dallas's apparent dependence on one region can't be judged until the missing states are filled in, so fixing that master data comes first.

**Limitation / extra data.** Region is the supplier's master address, and for Dallas it's mostly blank. A dependence finding would need complete addresses, ship-from locations and transport lead times per plant.